# 🤖 GigScore — Model Training

This notebook trains and evaluates all models:
1. Logistic Regression (baseline)
2. LightGBM with Optuna hyperparameter tuning
3. XGBoost with Optuna tuning
4. Stacking Ensemble

Evaluation: AUC-ROC, Gini, KS Statistic, F1, Precision, Recall

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import sys
sys.path.insert(0, '..')

from src.data.synthetic_generator import generate_synthetic_gig_data
from src.features.gig_features import engineer_gig_features
from src.features.india_features import engineer_india_features
from src.features.feature_pipeline import build_feature_pipeline, get_available_features, get_feature_names
from src.models.baseline import train_baseline
from src.models.lightgbm_model import train_lightgbm
from src.models.xgboost_model import train_xgboost
from src.models.ensemble import build_stacking_ensemble
from src.models.evaluator import evaluate_model, compare_models

In [ ]:
# Generate and prepare data
df = generate_synthetic_gig_data(n_samples=20000, seed=42)
df = engineer_gig_features(df)
df = engineer_india_features(df)

# Split
y = df['target']
num_f, cat_f, bin_f = get_available_features(df)
X = df[num_f + cat_f + bin_f]

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Pipeline (fit on train ONLY!)
pipeline = build_feature_pipeline(num_f, cat_f, bin_f)
X_tr = pipeline.fit_transform(X_train)
X_va = pipeline.transform(X_val)
X_te = pipeline.transform(X_test)

print(f'Train: {X_tr.shape}, Val: {X_va.shape}, Test: {X_te.shape}')

In [ ]:
# Train baseline
baseline = train_baseline(X_tr, y_train)
r1 = evaluate_model(baseline, X_te, y_test, 'Logistic Baseline')

In [ ]:
# Train LightGBM (use fewer trials for notebook demo)
lgbm = train_lightgbm(X_tr, y_train, X_va, y_val, n_trials=10)
r2 = evaluate_model(lgbm, X_te, y_test, 'LightGBM')

In [ ]:
# Train XGBoost
xgb = train_xgboost(X_tr, y_train, X_va, y_val, n_trials=10)
r3 = evaluate_model(xgb, X_te, y_test, 'XGBoost')

In [ ]:
# Stacking Ensemble
ens = build_stacking_ensemble(lgbm, xgb, baseline, X_tr, y_train)
r4 = evaluate_model(ens, X_te, y_test, 'Stacking Ensemble')

In [ ]:
# Final comparison
comparison = compare_models([r1, r2, r3, r4])
comparison